In [67]:
# Import libraries
import pandas as pd
import numpy as np
import os

In [68]:
# Define the path to the pickle file
path = 'C:\\Users\\User\\Documents\\Career Foundry\\Data Analytics Immersion\\Data Immersion 4. Python Fundamentals for Data Analysts\\Instacart Basket Analysis'

In [69]:
# Import ords_prods_merge
ords_prods_merge = pd.read_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'ords_prods_merge_updated_newcolumns.pkl'))

In [70]:
# 2 Find the aggregated mean of the “order_number” column grouped by “department_id” for the entire dataframe
ords_prods_merge.groupby('department_id').agg({'order_number': ['mean']})

,order_number
,mean
department_id,
1,15.457838
2,17.277920
3,17.170395
4,17.811403
5,15.215751
6,16.439806
7,17.225802
8,15.340650


# 3 Analyze the result. How do the results for the entire dataframe differ from those of the subset?
The subset and entire dataset share a consistent distribution pattern across departments, but the subset generally has slightly lower mean order_number values. This indicates that the subset may represent specific filtering criteria or a smaller sample of the overall data.

# 4 Follow the instructions in the Exercise for creating a loyalty flag for existing customers using the transform() and loc() functions. 
Since I already ran the code during the exercise and successfully created the loyalty_flag column, I will include the code in a markdown cell instead of re-running it. However, below, I will print out the loyalty_flag column to verify the results.
ords_prods_merge.loc[ords_prods_merge['max_order'] > 40, 'loyalty_flag'] = 'Loyal customer'
ords_prods_merge.loc[(ords_prods_merge['max_order'] <= 40) & (ords_prods_merge['max_order'] > 10), 'loyalty_flag'] = 'Regular customer'
ords_prods_merge.loc[ords_prods_merge['max_order'] <= 10, 'loyalty_flag'] = 'New customer'

In [73]:
# Check the loyalty_flag column
print(ords_prods_merge[['user_id', 'loyalty_flag']].head())

   user_id  loyalty_flag
0        1  New customer
1        1  New customer
2        1  New customer
3        1  New customer
4        1  New customer


# 5 The marketing team at Instacart wants to know whether there’s a difference between the spending habits of the three types of customers you identified. Use the loyalty flag you created and check the basic statistics of the product prices for each loyalty category (Loyal Customer, Regular Customer, and New Customer). What you’re trying to determine is whether the prices of products purchased by loyal customers differ from those purchased by regular or new customers.

In [75]:
# Check the columns that I have to determine which ones to use to answer the question
ords_prods_merge.columns

Index(['order_id', 'user_id', 'order_number', 'orders_day_of_week',
       'order_hour_of_day', 'days_since_prior_order', 'product_id',
       'add_to_cart_order', 'reordered', 'product_name', 'aisle_id',
       'department_id', 'prices', 'price_range_loc', 'busiest_day',
       'busiest_days_category', 'busiest_period_of_day', 'max_order',
       'loyalty_flag'],
      dtype='object')

In [76]:
# Check basic statistics for product prices grouped by loyalty_flag
ords_prods_merge.groupby('loyalty_flag')['prices'].describe()

,count,mean,std,min,25%,50%,75%,max
loyalty_flag,,,,,,,,
Loyal customer,10284093.0,10.386336,328.017787,1.0,4.2,7.4,11.2,99999.0
New customer,6243990.0,13.294670,597.560299,1.0,4.2,7.4,11.3,99999.0
Regular customer,15876776.0,12.495717,539.720919,1.0,4.2,7.4,11.3,99999.0


# What you’re trying to determine is whether the prices of products purchased by loyal customers differ from those purchased by regular or new customers.
New and regular customers tend to purchase more expensive items compared to loyal customers, possibly because they are exploring a variety of products, including higher-priced items. Loyal customers, on the other hand, exhibit a more predictable spending pattern, likely due to repeated purchases of familiar products. The identical maximum price across all groups suggests either a shared dataset anomaly or the presence of a high-priced item that skews the results. An investigation into whether outliers are affecting the results might be necessary. Despite the differences in average spending, the quartiles indicate that the majority of products purchased fall within the same price range (low- to mid-priced items) for all types of customers.

# 6  Create a spending flag for each user based on the average price across all their orders using the following criteria:
If the mean of the prices of products purchased by a user is lower than 10, then flag them as a “Low spender.”
If the mean of the prices of products purchased by a user is higher than or equal to 10, then flag them as a “High spender.”

In [79]:
# Calculate the mean price per user
ords_prods_merge['avg_price_per_user'] = ords_prods_merge.groupby('user_id')['prices'].transform('mean')

In [80]:
# Create spending flags based on the average price
ords_prods_merge.loc[ords_prods_merge['avg_price_per_user'] < 10, 'spending_flag'] = 'Low spender'
ords_prods_merge.loc[ords_prods_merge['avg_price_per_user'] >= 10, 'spending_flag'] = 'High spender'

In [81]:
# Count the result
ords_prods_merge['spending_flag'].value_counts()

spending_flag
Low spender     31770614
High spender      634245
Name: count, dtype: int64

# 7 Create an order frequency flag that marks the regularity of a user’s ordering behavior according to the median in the “days_since_prior_order” column. The criteria for the flag should be as follows:
If the median of “days_since_prior_order” is higher than 20, then the customer should be labeled a “Non-frequent customer.”
If the median is higher than 10 and lower than or equal to 20, then the customer should be labeled a “Regular customer.”
If the median is lower than or equal to 10, then the customer should be labeled a “Frequent customer.”

In [83]:
# Calculate the median of 'days_since_prior_order' for each user
ords_prods_merge['median_days_since_order'] = ords_prods_merge.groupby('user_id')['days_since_prior_order'].transform('median')

In [84]:
# Create order frequency flags based on the median
ords_prods_merge.loc[ords_prods_merge['median_days_since_order'] > 20, 'order_frequency_flag'] = 'Non-frequent customer'
ords_prods_merge.loc[(ords_prods_merge['median_days_since_order'] > 10) & (ords_prods_merge['median_days_since_order'] <= 20), 'order_frequency_flag'] = 'Regular customer'
ords_prods_merge.loc[ords_prods_merge['median_days_since_order'] <= 10, 'order_frequency_flag'] = 'Frequent customer'

In [85]:
# Count the number of customers in each order frequency category
ords_prods_merge['order_frequency_flag'].value_counts()

order_frequency_flag
Frequent customer        21559853
Regular customer          7208564
Non-frequent customer     3636437
Name: count, dtype: int64

In [86]:
# Check the dimension
ords_prods_merge.shape

(32404859, 23)

In [87]:
# Export the updated DataFrame as a pickle file
ords_prods_merge.to_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'ords_prods_merge_updated_with_flags.pkl'))